# Scenic Potential and Accessibility Analysis — GB, North KP, and AJK

This notebook computes the Scenic Potential Index (SPI) and Accessibility Index (AI)
for tehsils in Gilgit-Baltistan, northern Khyber Pakhtunkhwa, and Azad Jammu & Kashmir,
then runs the full statistical analysis described in the project proposal.

**Workflow:**
1. Load preprocessed, grid-aligned raster inputs from `data/interim/`
2. Compute SPI and AI rasters and save to `data/processed/`
3. Compute tehsil-level zonal statistics
4. Run descriptive statistics, spatial autocorrelation, OLS regression, priority ranking, and sensitivity analysis
5. Export final dataset to `outputs/` and `data/processed/`

Run all cells top to bottom — no manual edits required.

## 1. Imports and Setup

In [ ]:
import re
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from scipy.ndimage import uniform_filter, distance_transform_edt
from rasterio.features import rasterize, geometry_mask

warnings.filterwarnings('ignore')

try:
    from libpysal.weights import Queen
    from libpysal.weights.spatial_lag import lag_spatial
    from esda.moran import Moran, Moran_Local
    SPATIAL_BACKEND = 'libpysal'
except ImportError:
    SPATIAL_BACKEND = 'fallback'

try:
    import statsmodels.api as sm
    from statsmodels.stats.stattools import jarque_bera
    from statsmodels.stats.diagnostic import het_breuschpagan
    STATS_BACKEND = 'statsmodels'
except ImportError:
    STATS_BACKEND = 'fallback'

print(f'Spatial backend: {SPATIAL_BACKEND}')
print(f'Stats backend:   {STATS_BACKEND}')

In [ ]:
# Resolve project root regardless of where Jupyter was launched
cwd = Path.cwd()
if cwd.name == 'spi_gb_north':
    root = cwd
elif (cwd / 'spi_gb_north').exists():
    root = cwd / 'spi_gb_north'
elif (cwd / 'sds' / 'spi_gb_north').exists():
    root = cwd / 'sds' / 'spi_gb_north'
else:
    root = next((p for p in [cwd, *cwd.parents]
                 if (p / 'data').exists() and (p / 'outputs').exists()), cwd)

interim   = root / 'data' / 'interim'
processed = root / 'data' / 'processed'
outputs   = root / 'outputs'
processed.mkdir(parents=True, exist_ok=True)
outputs.mkdir(parents=True, exist_ok=True)

# SPI formula: SPI = 0.35*Relief + 0.25*Forest + 0.20*Water + 0.20*Snow
# Relief = Z(0.5*TPI_z + 0.5*TRI_z)
SPI_WEIGHTS = {'Relief': 0.35, 'Forest': 0.25, 'Water': 0.20, 'Snow': 0.20}
assert abs(sum(SPI_WEIGHTS.values()) - 1.0) < 1e-9

data_paths = {
    'dem':         interim / 'dem_32643_100m.tif',
    'forest_mask': interim / 'forest_mask_32643_100m.tif',
    'water_mask':  interim / 'water_mask_32643_100m.tif',
    'tpi_zscore':  interim / 'tpi_products' / 'tpi_zscore_radius2_32643_100m.tif',
    'tri':         interim / 'tri_32643_100m.tif',
    'snow_freq':   interim / 'snow_aligned_32643_100m' / 'snow_frequency_aligned_32643_100m.tif',
    'roads':       interim / 'roads_all_32643.gpkg',
    'tehsils':     interim / 'tehsils_aoi_32643.gpkg',
    'admin':       root / 'data' / 'raw' / 'admin_boundaries' / 'geoBoundaries-PAK-ADM3.geojson',
    'tehsil_list': root.parent / 'project' / 'aoi_tehsils_list.txt',
}

missing = [f'{k}: {v}' for k, v in data_paths.items()
           if k != 'tehsil_list' and not v.exists()]
if missing:
    raise FileNotFoundError('Missing inputs:\n' + '\n'.join(missing))
print(f'Root: {root}')
print('All input files found.')

## 2. Load Input Rasters

In [ ]:
def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        meta = {'crs': src.crs, 'transform': src.transform, 'nodata': src.nodata}
    return arr, meta

dem,    dem_meta = read_raster(data_paths['dem'])
tpi_z,  _        = read_raster(data_paths['tpi_zscore'])
forest, _        = read_raster(data_paths['forest_mask'])
water,  _        = read_raster(data_paths['water_mask'])
snow,   _        = read_raster(data_paths['snow_freq'])
tri_arr, _       = read_raster(data_paths['tri'])

# Normalize snow to [0,1] if it came in as 0-100
if np.nanmax(snow) > 1.5:
    snow /= 100.0

print(f'Grid shape: {dem.shape}  |  CRS: {dem_meta["crs"]}')

## 3. Scenic Potential Index (SPI)

$$\\text{SPI} = 0.35 \\cdot \\text{Relief} + 0.25 \\cdot \\text{Forest} + 0.20 \\cdot \\text{Water} + 0.20 \\cdot \\text{Snow}$$

Relief is the average of TPI z-score and TRI z-score, re-z-scored after combining. Forest and water are computed as neighborhood density over a 3 km window (30 pixels at 100 m) so a single isolated pixel does not score the same as a dense patch. Glacier pixels (snow frequency > 0.5) are stripped from the water mask to avoid double-counting.

In [ ]:
def zscore_arr(arr, mask):
    """Z-score over valid pixels; NaN outside mask."""
    vals = arr[mask]
    mu, sigma = vals.mean(), vals.std()
    if sigma < 1e-10:
        sigma = 1.0
    z = np.full_like(arr, np.nan)
    z[mask] = (arr[mask] - mu) / sigma
    return z

def neighborhood_density(binary, mask, window=30):
    """Fraction of valid pixels in a square window that are 1 (30 px = 3 km at 100 m)."""
    num = uniform_filter(np.where(mask, binary, 0.0), size=window, mode='nearest')
    den = uniform_filter(mask.astype(np.float32),     size=window, mode='nearest')
    out = np.where(den > 0, num / den, np.nan)
    out[~mask] = np.nan
    return out.astype(np.float32)

# Remove glaciers from water mask
water = np.where((snow > 0.5) & np.isfinite(water), 0.0, water)

# Valid-pixel mask: pixels that have data in all five input layers
valid = (np.isfinite(tpi_z) & np.isfinite(tri_arr) &
         np.isfinite(forest) & np.isfinite(water) & np.isfinite(snow))

# TRI: clip at P99.9 before z-scoring
tri_clip = np.clip(tri_arr, 0, np.nanpercentile(tri_arr[valid], 99.9))
tri_clip[~valid] = np.nan

# Component z-scores
tri_z    = zscore_arr(tri_clip, valid)
relief_z = zscore_arr(0.5 * tpi_z + 0.5 * tri_z, valid)
forest_z = zscore_arr(neighborhood_density(forest, valid), valid)
water_z  = zscore_arr(neighborhood_density(water,  valid), valid)
snow_z   = zscore_arr(snow, valid)

# Cap z-scores at ±3.5σ to prevent outlier pixels from dominating the weighted sum
for z in (relief_z, forest_z, water_z, snow_z):
    np.clip(z, -3.5, 3.5, out=z)
    z[~valid] = np.nan

spi = (SPI_WEIGHTS['Relief'] * relief_z +
       SPI_WEIGHTS['Forest'] * forest_z +
       SPI_WEIGHTS['Water']  * water_z  +
       SPI_WEIGHTS['Snow']   * snow_z)
spi[~valid] = np.nan

spi_meta = {
    'driver': 'GTiff', 'dtype': 'float32', 'count': 1,
    'width': spi.shape[1], 'height': spi.shape[0],
    'crs': dem_meta['crs'], 'transform': dem_meta['transform'],
    'nodata': np.nan, 'compress': 'lzw',
}
with rasterio.open(processed / 'spi_index.tif', 'w', **spi_meta) as dst:
    dst.write(spi, 1)

print(f'SPI range: [{np.nanmin(spi):.3f}, {np.nanmax(spi):.3f}]  mean={np.nanmean(spi):.3f}')
print(f'Valid pixels: {valid.sum():,}  ({valid.sum() / valid.size * 100:.1f}% of bounding box)')
print(f'Saved: {processed / "spi_index.tif"}')

## 4. Accessibility Index (AI)

AI uses exponential decay from road proximity, adjusted for terrain slope:

$$\\text{AI} = e^{-\\lambda \\cdot t}$$

where $t$ is travel time to the nearest road pixel and $\\lambda = \\ln(2)/90$ gives a half-life of 90 minutes. Road speed is taken from OSM highway tags and reduced by a slope modifier (slope > 5°: ×0.8, > 15°: ×0.6, > 25°: ×0.4).

In [ ]:
# Slope from DEM
px = abs(dem_meta['transform'].a)
dy, dx = np.gradient(np.where(valid, dem, np.nan), px, px)
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
slope_deg[~valid] = np.nan

slope_mod = np.ones_like(slope_deg)
slope_mod[slope_deg > 5]  = 0.8
slope_mod[slope_deg > 15] = 0.6
slope_mod[slope_deg > 25] = 0.4
slope_mod[~valid] = np.nan

# Rasterize road network with per-type speeds
base_speeds = {
    'motorway': 80, 'trunk': 60, 'primary': 45, 'secondary': 35,
    'tertiary': 25, 'unclassified': 20, 'residential': 15, 'track': 10,
}
roads_gdf = gpd.read_file(data_paths['roads'])
road_type_col = next((c for c in ['highway', 'road_type', 'fclass']
                      if c in roads_gdf.columns), None)
if road_type_col:
    roads_gdf['_spd'] = (roads_gdf[road_type_col]
                         .str.lower().map(base_speeds).fillna(15).astype(np.float32))
else:
    roads_gdf['_spd'] = 15.0

speed_raster = np.full(dem.shape, 3.0, dtype=np.float32)
for spd, grp in roads_gdf.groupby('_spd', sort=True):
    shapes = [(g, spd) for g in grp.geometry if g is not None and not g.is_empty]
    if shapes:
        burned = rasterize(shapes, out_shape=dem.shape,
                           transform=dem_meta['transform'],
                           fill=0, dtype=np.float32, all_touched=True)
        speed_raster[burned > 0] = burned[burned > 0]

adj_speed = np.clip(speed_raster * slope_mod, 1.0, None)
adj_speed[~valid] = np.nan

road_mask = (speed_raster > 3.0) & valid
dist_m    = distance_transform_edt(~road_mask, sampling=(px, px)).astype(np.float32)
spd_mpm   = adj_speed * (1000.0 / 60.0)   # km/h -> m/min
travel_t  = dist_m / np.where(spd_mpm > 0, spd_mpm, np.nan)
travel_t[~valid] = np.nan

lam = np.log(2) / 90.0   # half-life = 90 min
ai  = np.exp(-lam * travel_t).astype(np.float32)
ai[~valid] = np.nan

ai_meta = spi_meta.copy()
with rasterio.open(processed / 'ai_index.tif', 'w', **ai_meta) as dst:
    dst.write(ai, 1)

print(f'Road pixels in AOI: {road_mask.sum():,} ({road_mask.sum() / valid.sum() * 100:.2f}%)')
print(f'AI range: [{np.nanmin(ai):.4f}, {np.nanmax(ai):.4f}]  mean={np.nanmean(ai):.4f}')
print(f'Saved: {processed / "ai_index.tif"}')

## 5. SPI and AI Raster Maps

These maps show the pixel-level outputs before aggregating to tehsil level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cmap_spi = plt.cm.RdYlGn.copy(); cmap_spi.set_bad('white')
cmap_ai  = plt.cm.YlOrRd.copy(); cmap_ai.set_bad('white')

ax = axes[0]
im = ax.imshow(np.where(valid, spi, np.nan), cmap=cmap_spi,
               vmin=np.nanpercentile(spi[valid], 2),
               vmax=np.nanpercentile(spi[valid], 98))
ax.set_title('Scenic Potential Index (SPI)', fontweight='bold')
ax.set_axis_off()
plt.colorbar(im, ax=ax, shrink=0.75)

ax = axes[1]
im = ax.imshow(np.where(valid, ai, np.nan), cmap=cmap_ai, vmin=0, vmax=1)
ax.set_title('Accessibility Index (AI)', fontweight='bold')
ax.set_axis_off()
plt.colorbar(im, ax=ax, shrink=0.75)

plt.tight_layout()
plt.savefig(outputs / '00_spi_ai_rasters.png', bbox_inches='tight', facecolor='white', dpi=150)
plt.show()

## 6. Tehsil Zonal Statistics

For each tehsil polygon the mean SPI and AI are extracted from the rasters. The admin boundary file is filtered to the study-area tehsils using a name list; if fewer than 20 names match, the preprocessed tehsil layer is used instead.

In [ ]:
def norm_name(s):
    s = str(s).upper().strip()
    return re.sub(r'[^A-Z0-9]+', ' ', s.replace('`', '')).strip()

def load_name_list(path):
    if not path.exists():
        return set()
    text = path.read_text().replace('\\n', '\n')
    return {norm_name(l) for l in text.splitlines()
            if l.strip() and not l.lower().startswith('names of')}

AJK_TEHSILS = {
    'BAGH', 'DHEERKOT', 'BARNALA', 'BHIMBER', 'SAMAHNI', 'HATTIAN BALA',
    'HAVELI', 'KOTLI', 'NAKIAL', 'SEHNSA', 'DUDYAL', 'MIRPUR',
    'MUZAFFARABAD', 'ATHUMQAM', 'ABBASPUR', 'HAJEERA', 'RAWALAKOT', 'PALLANDARI',
}

aoi_names = load_name_list(data_paths['tehsil_list']) | {norm_name(x) for x in AJK_TEHSILS}

admin = gpd.read_file(data_paths['admin'])
if admin.crs is None:
    admin = admin.set_crs('EPSG:4326')
admin['geometry'] = admin.geometry.make_valid()
admin = admin[~admin.geometry.is_empty]
admin['_key'] = admin['shapeName'].map(norm_name)

tehsils = admin[admin['_key'].isin(aoi_names)].copy() if aoi_names else None
if tehsils is None or len(tehsils) < 20:
    print('Name list too small — using preprocessed tehsil layer')
    tehsils = gpd.read_file(data_paths['tehsils'])
tehsils = tehsils.to_crs('EPSG:32643')

name_col = next((c for c in ['shapeName', 'NAME', 'TEHSIL', 'tehsil', 'NAME_EN']
                 if c in tehsils.columns), None)
print(f'Tehsils: {len(tehsils)}  |  name column: {name_col}')

def zonal_mean(raster, zones, label):
    rows = []
    for _, row in zones.iterrows():
        name = row[name_col] if name_col else str(row.name)
        if row.geometry is None or row.geometry.is_empty:
            rows.append({'tehsil': name, f'{label}_mean': np.nan, f'{label}_count': 0})
            continue
        mask = geometry_mask([row.geometry], out_shape=dem.shape,
                             transform=dem_meta['transform'], invert=True)
        vals = raster[mask & np.isfinite(raster)]
        rows.append({'tehsil': name,
                     f'{label}_mean':  float(np.nanmean(vals)) if vals.size else np.nan,
                     f'{label}_count': int(vals.size)})
    return pd.DataFrame(rows)

spi_zonal = zonal_mean(spi, tehsils, 'spi')
ai_zonal  = zonal_mean(ai,  tehsils, 'ai')
zonal = spi_zonal.merge(ai_zonal, on='tehsil', how='outer')

zonal_path = processed / 'tehsil_spi_ai_zonal_stats.csv'
zonal.to_csv(zonal_path, index=False)
print(f'Saved: {zonal_path}')
print(f'Tehsils with SPI data: {(zonal.spi_count > 0).sum()}')

---
## 7. Tehsil-Level Statistical Analysis

The sections below re-read the zonal CSV from disk so this half of the notebook
can be re-run independently of the raster computation above.

In [ ]:
gdf   = gpd.read_file(data_paths['tehsils'])
zonal = pd.read_csv(processed / 'tehsil_spi_ai_zonal_stats.csv')

gdf['_key']   = gdf.iloc[:, 0].map(norm_name) if 'shapeName' not in gdf.columns \
                else gdf['shapeName'].map(norm_name)
zonal['_key'] = zonal['tehsil'].map(norm_name)

agdf = gdf.merge(zonal, on='_key', how='left')
agdf['tehsil'] = agdf['tehsil'].fillna(agdf.get('shapeName', agdf.iloc[:, 0]))
agdf = agdf.dropna(subset=['spi_mean', 'ai_mean']).copy()
agdf['area_km2'] = agdf.to_crs(agdf.crs).geometry.area / 1e6
agdf['spi'] = agdf['spi_mean']
agdf['ai']  = agdf['ai_mean']

def zscore(s):
    s  = pd.Series(s, dtype='float64')
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd > 0 else pd.Series(np.zeros(len(s)), index=s.index)

agdf['spi_z']     = zscore(agdf['spi'])
agdf['ai_z']      = zscore(agdf['ai'])
agdf['gap_index'] = agdf['spi_z'] - agdf['ai_z']
agdf['gap_rank']  = agdf['gap_index'].rank(ascending=False, method='min').astype(int)

def lisa_labels(local, alpha):
    quad_map = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
    return [quad_map.get(int(q), 'NS') if p <= alpha else 'NS'
            for q, p in zip(local.q, local.p_sim)]

def savefig(name):
    plt.savefig(outputs / name, bbox_inches='tight', facecolor='white')
    print(f'Saved: {outputs / name}')

print(f'Tehsils in analysis: {len(agdf)}')
display(agdf[['tehsil', 'spi', 'ai', 'gap_index']]
        .sort_values('gap_index', ascending=False).head(10))

### 7.1 Descriptive Statistics

In [ ]:
summary = agdf[['spi', 'ai', 'gap_index', 'area_km2']].describe().T
summary.to_csv(outputs / 'descriptive_statistics.csv')
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title, color in [
    (axes[0], 'spi',       'SPI tehsil mean',   '#2E7D32'),
    (axes[1], 'ai',        'AI tehsil mean',    '#C55A11'),
    (axes[2], 'gap_index', 'Scenic-access gap', '#6A51A3'),
]:
    ax.hist(agdf[col].dropna(), bins=20, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(agdf[col].mean(), color='black', lw=1.2, ls='--', label='mean')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
savefig('01_distributions.png')
plt.show()

### 7.2 Choropleth Maps

The gap index is high where scenic potential is above average and accessibility is below average.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, col, title, cmap in [
    (axes[0], 'spi',       'Scenic Potential Index (SPI)',     'viridis'),
    (axes[1], 'ai',        'Accessibility Index (AI)',         'YlOrRd'),
    (axes[2], 'gap_index', 'Scenic-Access Gap (SPI z - AI z)', 'RdYlGn'),
]:
    agdf.plot(column=col, cmap=cmap, linewidth=0.25, edgecolor='0.35', legend=True, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_axis_off()
plt.tight_layout()
savefig('02_choropleth_maps.png')
plt.show()

### 7.3 Queen Contiguity Spatial Weights

In [ ]:
agdf = agdf.reset_index(drop=True)
w = Queen.from_dataframe(agdf, use_index=False)
if w.islands:
    print(f'Isolated units (no neighbors): {w.islands}')
w.transform = 'r'

agdf['n_neighbors'] = [len(w.neighbors[i]) for i in range(len(agdf))]
display(agdf['n_neighbors'].describe().to_frame('queen_neighbors').T)

### 7.4 Global Spatial Autocorrelation — Moran's I

In [ ]:
rows = []
for label, col in [('SPI', 'spi'), ('AI', 'ai'), ('Gap', 'gap_index')]:
    m = Moran(agdf[col].to_numpy(), w, permutations=999, two_tailed=True)
    rows.append({'variable': label, 'I': m.I, 'E[I]': m.EI, 'z': m.z_sim, 'p': m.p_sim})
moran_df = pd.DataFrame(rows)
moran_df.to_csv(outputs / 'global_moran.csv', index=False)
display(moran_df.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(moran_df['variable'], moran_df['I'],
       color=['#2E7D32', '#C55A11', '#6A51A3'], alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel("Moran's I")
ax.set_title('Global Spatial Autocorrelation')
plt.tight_layout()
savefig('03_moran_i.png')
plt.show()

### 7.5 Local Spatial Clustering — LISA

Local Moran's I classifies each tehsil as HH, LL, HL, LH, or not significant (NS). Threshold: p < 0.10, which is appropriate for moderate sample sizes.

In [ ]:
ALPHA = 0.10

for col, prefix in [('spi', 'spi'), ('ai', 'ai'), ('gap_index', 'gap')]:
    loc = Moran_Local(agdf[col].to_numpy(), w, permutations=999)
    agdf[f'{prefix}_cluster'] = lisa_labels(loc, ALPHA)
    agdf[f'{prefix}_local_I'] = loc.Is
    agdf[f'{prefix}_local_p'] = loc.p_sim

cluster_colors = {'HH': '#d7191c', 'HL': '#fdae61',
                  'LH': '#abd9e9', 'LL': '#2c7bb6', 'NS': '#d3d3d3'}

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, prefix, title in [
    (axes[0], 'spi', 'SPI LISA Clusters'),
    (axes[1], 'ai',  'AI LISA Clusters'),
    (axes[2], 'gap', 'Gap LISA Clusters'),
]:
    agdf.plot(color=agdf[f'{prefix}_cluster'].map(cluster_colors),
              linewidth=0.25, edgecolor='0.35', ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_axis_off()
    handles = [Patch(facecolor=c, label=l)
               for l, c in cluster_colors.items()
               if l in set(agdf[f'{prefix}_cluster'])]
    ax.legend(handles=handles, loc='lower right', fontsize=8)
plt.tight_layout()
savefig('04_lisa_clusters.png')
plt.show()

for prefix, label in [('spi', 'SPI'), ('ai', 'AI'), ('gap', 'Gap')]:
    print(f'{label}: {agdf[f"{prefix}_cluster"].value_counts().to_dict()}')

### 7.6 Regression: SPI ~ AI

OLS regression tests whether accessibility meaningfully predicts scenic potential variation. Residuals are checked for spatial autocorrelation (Moran's I), heteroskedasticity (Breusch-Pagan), and normality (Jarque-Bera).

In [ ]:
X = sm.add_constant(agdf['ai'].astype(float))
y = agdf['spi'].astype(float)
ols = sm.OLS(y, X).fit()

resid_moran = Moran(ols.resid.to_numpy(), w, permutations=999)
bp_lm, bp_p, _, _ = het_breuschpagan(ols.resid, ols.model.exog)
jb_stat, jb_p, _, _ = jarque_bera(ols.resid)

reg_summary = pd.DataFrame([
    ('n',                 len(agdf)),
    ('intercept',         ols.params['const']),
    ('ai_coef',           ols.params['ai']),
    ('ai_p',              ols.pvalues['ai']),
    ('R2',                ols.rsquared),
    ('adj_R2',            ols.rsquared_adj),
    ('residual_moran_I',  resid_moran.I),
    ('residual_moran_p',  resid_moran.p_sim),
    ('breusch_pagan_p',   bp_p),
    ('jarque_bera_p',     jb_p),
], columns=['metric', 'value'])
reg_summary.to_csv(outputs / 'regression_summary.csv', index=False)
display(reg_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(agdf['ai'], agdf['spi'], s=40, alpha=0.8, color='#2B6CB0', edgecolor='white')
x_line = np.linspace(agdf['ai'].min(), agdf['ai'].max(), 100)
ax.plot(x_line, ols.params['const'] + ols.params['ai'] * x_line, color='black', lw=1.5)
ax.set_title(f'SPI ~ AI  (R² = {ols.rsquared:.3f})')
ax.set_xlabel('AI tehsil mean'); ax.set_ylabel('SPI tehsil mean')

ax = axes[1]
ax.scatter(ols.fittedvalues, ols.resid, s=35, alpha=0.8, color='#6A51A3', edgecolor='white')
ax.axhline(0, color='black', lw=1)
ax.set_title('Residuals vs Fitted')
ax.set_xlabel('Fitted SPI'); ax.set_ylabel('Residual')

plt.tight_layout()
savefig('05_regression.png')
plt.show()

### 7.7 Priority Regions

Two criteria identify tehsils that warrant infrastructure investment:
- **Strict**: SPI in the top quartile *and* AI in the bottom quartile
- **Relaxed**: SPI above average *and* AI below average

In [ ]:
spi_q75 = agdf['spi_z'].quantile(0.75)
ai_q25  = agdf['ai_z'].quantile(0.25)

agdf['priority_strict']  = (agdf['spi_z'] >= spi_q75) & (agdf['ai_z'] <= ai_q25)
agdf['priority_relaxed'] = (agdf['spi_z'] > 0) & (agdf['ai_z'] < 0)
agdf['priority_lisa']    = (agdf['gap_cluster'].isin(['HH', 'HL']) &
                            (agdf['gap_local_p'] <= ALPHA))

priority_cols = [
    'tehsil', 'spi', 'ai', 'spi_z', 'ai_z', 'gap_index', 'gap_rank',
    'priority_strict', 'priority_relaxed', 'priority_lisa', 'spi_cluster', 'gap_cluster',
]
priority_table = agdf[priority_cols].sort_values('gap_index', ascending=False)
priority_table.to_csv(outputs / 'priority_regions.csv', index=False)

print(f'Strict priority:  {agdf.priority_strict.sum()} tehsils')
print(f'Relaxed priority: {agdf.priority_relaxed.sum()} tehsils')
display(priority_table.head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(agdf['ai_z'], agdf['spi_z'], color='#bdbdbd', s=35, edgecolor='white', label='Other')
mask_r = agdf['priority_relaxed'] & ~agdf['priority_strict']
ax.scatter(agdf.loc[mask_r, 'ai_z'], agdf.loc[mask_r, 'spi_z'],
           color='#fdae61', s=55, edgecolor='k', lw=0.4, label='Relaxed')
ax.scatter(agdf.loc[agdf['priority_strict'], 'ai_z'],
           agdf.loc[agdf['priority_strict'], 'spi_z'],
           color='#d7191c', s=70, edgecolor='k', lw=0.5, label='Strict')
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.axhline(spi_q75, color='#d7191c', ls='--', lw=1)
ax.axvline(ai_q25,  color='#d7191c', ls='--', lw=1)
ax.set_xlabel('AI z-score'); ax.set_ylabel('SPI z-score')
ax.set_title('Priority Quadrant'); ax.legend()

ax = axes[1]
agdf.plot(color=agdf['priority_strict'].map({True: '#d7191c', False: '#bdbdbd'}),
          linewidth=0.25, edgecolor='0.35', ax=ax)
ax.set_title('Strict Priority Tehsils', fontweight='bold'); ax.set_axis_off()

plt.tight_layout()
savefig('06_priority_regions.png')
plt.show()

### 7.8 SPI Component Contributions

Tehsil-level means for TRI, forest, water, and snow are extracted from the raw input rasters and correlated with tehsil SPI to show which landscape features drive scenic potential.

In [ ]:
comp_raster_paths = {
    'tri_mean':   data_paths['tri'],
    'forest_pct': data_paths['forest_mask'],
    'water_pct':  data_paths['water_mask'],
    'snow_pct':   data_paths['snow_freq'],
}

def raster_zonal_mean(raster_path, zones, label):
    rows = []
    with rasterio.open(raster_path) as src:
        arr = src.read(1).astype('float32')
        nd = src.nodata
        if nd is not None and np.isfinite(nd):
            arr[arr == nd] = np.nan
        crs, tf = src.crs, src.transform
    zones_proj = zones.to_crs(crs) if zones.crs != crs else zones
    for _, row in zones_proj.iterrows():
        name = row[name_col] if name_col else str(row.name)
        mask = geometry_mask([row.geometry], out_shape=arr.shape, transform=tf, invert=True)
        vals = arr[mask & np.isfinite(arr)]
        rows.append({'tehsil': name, label: float(np.nanmean(vals)) if vals.size else np.nan})
    return pd.DataFrame(rows)

comp_df = None
for label, path in comp_raster_paths.items():
    df = raster_zonal_mean(path, tehsils, label)
    comp_df = df if comp_df is None else comp_df.merge(df, on='tehsil', how='outer')

agdf = agdf.merge(comp_df, on='tehsil', how='left')
component_cols = [c for c in ['tri_mean', 'forest_pct', 'water_pct', 'snow_pct']
                  if c in agdf.columns]

corr_rows = []
for col in component_cols:
    valid_c = agdf[[col, 'spi']].dropna()
    if len(valid_c) < 5:
        continue
    Xc = sm.add_constant(valid_c[col].astype(float))
    cm = sm.OLS(valid_c['spi'].astype(float), Xc).fit()
    corr_rows.append({
        'component':   col,
        'pearson_r':   valid_c.corr()[col]['spi'],
        'ols_slope':   cm.params[col],
        'ols_slope_p': cm.pvalues[col],
        'R2':           cm.rsquared,
    })
comp_corr = pd.DataFrame(corr_rows)
comp_corr.to_csv(outputs / 'component_correlations.csv', index=False)
display(comp_corr)

### 7.9 Sensitivity Analysis

Four alternative weighting schemes are tested at the tehsil level to check whether priority rankings are robust to the choice of SPI weights. Relief weight (0.35) is approximated by TRI since TPI zonal means are not extracted here.

In [ ]:
for col in component_cols:
    agdf[f'{col}_z'] = zscore(agdf[col])

weight_sets = {
    'proposal':         {'tri_mean_z': 0.35, 'forest_pct_z': 0.25, 'water_pct_z': 0.20, 'snow_pct_z': 0.20},
    'equal':            {'tri_mean_z': 0.25, 'forest_pct_z': 0.25, 'water_pct_z': 0.25, 'snow_pct_z': 0.25},
    'terrain_emphasis': {'tri_mean_z': 0.50, 'forest_pct_z': 0.20, 'water_pct_z': 0.15, 'snow_pct_z': 0.15},
    'veg_water_emph':   {'tri_mean_z': 0.30, 'forest_pct_z': 0.30, 'water_pct_z': 0.25, 'snow_pct_z': 0.15},
}

sens_rows = []
priority_freq = np.zeros(len(agdf))
for name, wts in weight_sets.items():
    z_cols = [c for c in wts if c in agdf.columns]
    if not z_cols:
        continue
    agdf[f'spi_alt_{name}'] = sum(agdf[c] * wts[c] for c in z_cols)
    corr = agdf[['spi', f'spi_alt_{name}']].dropna().corr().iloc[0, 1]
    alt_z = zscore(agdf[f'spi_alt_{name}'])
    priority_freq += ((alt_z >= alt_z.quantile(0.75)) & agdf['priority_strict']).astype(float)
    sens_rows.append({'weight_set': name, 'corr_with_current_spi': corr})

agdf['priority_frequency'] = priority_freq
sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(outputs / 'sensitivity_analysis.csv', index=False)
display(sens_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(sens_df['weight_set'], sens_df['corr_with_current_spi'],
            color='#2E7D32', alpha=0.85)
axes[0].set_ylim(0, 1); axes[0].set_ylabel('Pearson r with current SPI')
axes[0].set_title('SPI Weighting Sensitivity')
axes[0].tick_params(axis='x', rotation=20)
agdf.plot(column='priority_frequency', cmap='YlOrRd', linewidth=0.25,
          edgecolor='0.35', legend=True, ax=axes[1])
axes[1].set_title('Priority Stability Across Weighting Schemes')
axes[1].set_axis_off()
plt.tight_layout()
savefig('07_sensitivity.png')
plt.show()

### 7.10 Bivariate SPI-AI Map

In [ ]:
def tertile(s):
    q1, q2 = s.quantile([1/3, 2/3])
    return pd.cut(s, bins=[-np.inf, q1, q2, np.inf], labels=['Low', 'Med', 'High'])

agdf['spi_class'] = tertile(agdf['spi'])
agdf['ai_class']  = tertile(agdf['ai'])
agdf['biv_class'] = agdf['spi_class'].astype(str) + ' SPI / ' + agdf['ai_class'].astype(str) + ' AI'

bivar_colors = {
    'Low SPI / Low AI':  '#d9f0d3', 'Med SPI / Low AI':  '#addd8e', 'High SPI / Low AI':  '#31a354',
    'Low SPI / Med AI':  '#fdd0a2', 'Med SPI / Med AI':  '#bdbdbd', 'High SPI / Med AI':  '#756bb1',
    'Low SPI / High AI': '#ef3b2c', 'Med SPI / High AI': '#fd8d3c', 'High SPI / High AI': '#54278f',
}

fig, ax = plt.subplots(figsize=(10, 8))
agdf.plot(color=agdf['biv_class'].map(bivar_colors).fillna('#cccccc'),
          linewidth=0.25, edgecolor='0.35', ax=ax)
ax.set_title('Bivariate SPI-AI Classification', fontweight='bold')
ax.set_axis_off()
handles = [Patch(facecolor=c, edgecolor='0.35', label=l)
           for l, c in bivar_colors.items() if l in set(agdf['biv_class'])]
ax.legend(handles=handles, loc='lower right', fontsize=8, ncol=2)
plt.tight_layout()
savefig('08_bivariate_map.png')
plt.show()

## 8. Export Final Dataset

In [ ]:
out_geojson  = outputs   / 'tehsil_spi_ai_analysis.geojson'
out_csv      = outputs   / 'tehsil_spi_ai_analysis.csv'
proc_geojson = processed / 'final_tehsil_spi_ai_analysis.geojson'
proc_csv     = processed / 'final_tehsil_spi_ai_analysis.csv'

agdf.to_file(out_geojson,  driver='GeoJSON')
agdf.to_file(proc_geojson, driver='GeoJSON')
agdf.drop(columns='geometry').to_csv(out_csv,  index=False)
agdf.drop(columns='geometry').to_csv(proc_csv, index=False)

report = {
    'n_tehsils':        int(len(agdf)),
    'spi_mean':         float(agdf['spi'].mean()),
    'ai_mean':          float(agdf['ai'].mean()),
    'global_moran':     moran_df.to_dict(orient='records'),
    'priority_strict':  int(agdf['priority_strict'].sum()),
    'priority_relaxed': int(agdf['priority_relaxed'].sum()),
    'outputs_dir':      str(outputs),
}
with open(outputs / 'analysis_summary.json', 'w') as f:
    json.dump(report, f, indent=2)

print('Exports complete:')
for p in [out_csv, proc_csv, out_geojson, proc_geojson,
          outputs / 'analysis_summary.json']:
    print(f'  {p}')